# Retrieval v2 – Stage 6 Candidate Construction

Run the frozen Stage 1–5 ASR-text retriever over all 40 Extension40 queries against All50, evaluate candidate coverage for the predefined K/M policies, and produce the fixed candidate artifact for Stage 7. No reranker is loaded in this notebook.


In [1]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, subprocess, sys
from pathlib import Path
import pandas as pd

CODE_ROOT     = Path("/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison")
ARTIFACT_ROOT = CODE_ROOT
CONFIG_PATH   = CODE_ROOT / "configs" / "retrieval_v2.json"
RUNNER_PATH   = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CONFIG"] = str(CONFIG_PATH)
os.chdir(CODE_ROOT)

print("Code root:    ", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Config:       ", CONFIG_PATH)


Mounted at /content/drive
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Config:        /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/configs/retrieval_v2.json


In [2]:
config = json.loads(CONFIG_PATH.read_text(encoding = "utf-8"))
stage = config["stage06_candidate_construction"]
print("Frozen Stage 1–5 source")
print("  channel     :", stage["source"]["channel_id"])
print("  lexical     :", stage["source"]["sparse_method"])
print("  dense       :", stage["source"]["dense_method"])
print("  hybrid      :", stage["source"]["hybrid_method"])
print("  aggregation :", stage["source"]["video_aggregation"])
print("  queries     :", stage["query_set"])
print("  corpus      :", stage["corpus"])
print("  K grid      :", stage["candidate_video_k"])
print("  M grid      :", stage["windows_per_video_m"])


Frozen Stage 1–5 source
  channel     : parakeet_processed
  lexical     : L2_bm25_preserving
  dense       : D1_e5_large_instruct
  hybrid      : H3_norm_25_75
  aggregation : P0_max
  queries     : extension40_full
  corpus      : all50
  K grid      : [10, 20, 30]
  M grid      : [1, 3, 5]


In [3]:
# Run Stage 6 with live progress + full traceback
import os
import subprocess
import time

command = [sys.executable, "-u", str(RUNNER_PATH), "--stage", "stage06_candidate_construction"]
environment = os.environ.copy()
environment["PYTHONUNBUFFERED"] = "1"

print("=" * 100, flush = True)
print("STAGE 6 – CANDIDATE CONSTRUCTION", flush = True)
print("=" * 100, flush = True)
print("Command :", " ".join(command), flush = True)
print("CWD     :", CODE_ROOT, flush = True)
print("Config  :", CONFIG_PATH, flush = True)
print("Artifacts:", ARTIFACT_ROOT, flush = True)
print("=" * 100, flush = True)

started = time.perf_counter()
process = subprocess.Popen(
    command,
    cwd = str(CODE_ROOT),
    env = environment,
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
    bufsize = 1,
)

assert process.stdout is not None
for line in iter(process.stdout.readline, "") :
    print(line, end = "", flush = True)

process.stdout.close()
return_code = process.wait()
elapsed = time.perf_counter() - started

print("\n" + "=" * 100, flush = True)
print(f"Stage 6 finished in {elapsed:.1f}s", flush = True)
print(f"Exit code: {return_code}", flush = True)
print("=" * 100, flush = True)

if (return_code != 0) :
    raise RuntimeError(
        f"Stage 6 failed with exit code {return_code}. "
        "The complete runner error is visible directly above."
    )

print("STAGE 6 COMPLETE", flush = True)


STAGE 6 – CANDIDATE CONSTRUCTION
Command : /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage06_candidate_construction
CWD     : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Config  : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/configs/retrieval_v2.json
Artifacts: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
AIC 2026 RETRIEVAL V2 – STAGE 6 CANDIDATE CONSTRUCTION
Query set:     extension40_full (Extension40 full 40)
Corpus:        all50 (All50)
Source:        parakeet_processed / H3_norm_25_75

      D1_e5_large_instruct/extension40_full: query cache MISS; reusing 20 frozen Stage 2 development embeddings and encoding only 20 newly exposed queries
      D1_e5_large_instruct: loading intfloat/multilingual-e5-larg

In [4]:
REPORT_ROOT = ARTIFACT_ROOT / "reports" / "retrieval_v2" / "stage06_candidate_construction"
summary = json.loads((REPORT_ROOT / "stage_summary.json").read_text(encoding = "utf-8"))
policies = pd.read_csv(REPORT_ROOT / "candidate_policy_summary.csv")

print("Stage 6 status:", "PASS" if summary["passed"] else "FAIL")
print("Recommended:  ", summary["recommended_policy_id"])
display(policies.sort_values(["joint_candidate_recall", "candidate_pairs_max"], ascending = [False, True]))


Stage 6 status: PASS
Recommended:   K30_M5


,policy_id,video_k,windows_per_video_m,query_count,candidate_pairs_mean,candidate_pairs_max,video_candidate_recall,temporal_candidate_recall,conditional_temporal_recall,joint_candidate_recall,video_miss_count,temporal_miss_count,joint_miss_count,video_oracle_recall,temporal_oracle_recall,joint_oracle_recall,recommended
8,K30_M5,30,5,40,149.275,150,1.000,0.975,0.975000,0.975,0,1,1,1.000,0.975,0.975,True
11,K40_M5,40,5,40,197.500,200,1.000,0.975,0.975000,0.975,0,1,1,1.000,0.975,0.975,False
5,K20_M5,20,5,40,99.425,100,0.950,0.975,1.000000,0.950,2,1,2,0.950,0.975,0.950,False
2,K10_M5,10,5,40,49.600,50,0.925,0.975,1.000000,0.925,3,1,3,0.925,0.975,0.925,False
7,K30_M3,30,3,40,89.925,90,1.000,0.925,0.925000,0.925,0,3,3,1.000,0.925,0.925,False
10,K40_M3,40,3,40,119.600,120,1.000,0.925,0.925000,0.925,0,3,3,1.000,0.925,0.925,False
4,K20_M3,20,3,40,59.950,60,0.950,0.925,0.947368,0.900,2,3,4,0.950,0.925,0.900,False
1,K10_M3,10,3,40,29.950,30,0.925,0.925,0.945946,0.875,3,3,5,0.925,0.925,0.875,False
3,K20_M1,20,1,40,20.000,20,0.950,0.850,0.894737,0.850,2,6,6,0.950,0.850,0.850,False
6,K30_M1,30,1,40,30.000,30,1.000,0.850,0.850000,0.850,0,6,6,1.000,0.850,0.850,False


In [5]:
failures = pd.read_csv(REPORT_ROOT / "candidate_failures.csv")
metrics  = pd.read_csv(REPORT_ROOT / "candidate_metrics.csv")

print("Diagnostic slices")
display(metrics[metrics["slice"].isin(["all40", "split:development20", "split:holdout20"])])

print("Candidate failures")
display(failures if len(failures) else pd.DataFrame({"status" : ["No candidate failures for evaluated policies"]}))


Diagnostic slices


,policy_id,slice,query_count,video_candidate_recall,temporal_candidate_recall,joint_candidate_recall,candidate_pairs_mean,candidate_pairs_max
0,K10_M1,all40,40,0.925,0.850,0.825,10.000,10
1,K10_M1,split:development20,20,0.900,0.800,0.800,10.000,10
2,K10_M1,split:holdout20,20,0.950,0.900,0.850,10.000,10
5,K10_M3,all40,40,0.925,0.925,0.875,29.950,30
6,K10_M3,split:development20,20,0.900,0.850,0.800,29.950,30
7,K10_M3,split:holdout20,20,0.950,1.000,0.950,29.950,30
10,K10_M5,all40,40,0.925,0.975,0.925,49.600,50
11,K10_M5,split:development20,20,0.900,0.950,0.900,49.450,50
12,K10_M5,split:holdout20,20,0.950,1.000,0.950,49.750,50
15,K20_M1,all40,40,0.950,0.850,0.850,20.000,20


Candidate failures


,policy_id,video_k,windows_per_video_m,query_id,query_text,historical_split,task_type,query_category,difficulty,correct_video,correct_video_rank,correct_video_retained,relevant_window_retained,joint_success,candidate_pair_count,candidate_video_count,failure_type
0,K10_M1,10,1,R2-15,Một người đang chụp hình những chiếc ván trượt...,holdout20,KIS,other,medium,K01_V018,14,False,True,False,10,10,CANDIDATE_VIDEO_MISS
1,K10_M1,10,1,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,development20,KIS,other,easy,K03_V019,24,False,False,False,10,10,CANDIDATE_VIDEO_MISS
2,K10_M1,10,1,R2-8,Đoạn clip mở đầu với hình ảnh cây cầu có hệ th...,development20,KIS,number_or_measurement,medium,K03_V023,27,False,False,False,10,10,CANDIDATE_VIDEO_MISS
3,K10_M1,10,1,R1-13,Đoạn video mô tả một người ngồi vệ sinh máy ản...,development20,KIS,other,medium,L30_V095,1,True,False,False,10,10,CANDIDATE_WINDOW_MISS
4,K10_M1,10,1,R1-9,Đoạn clip là cảnh thu hoạch dứa ở miền Tây: mộ...,development20,KIS,person_organization_location,medium,L27_V013,1,True,False,False,10,10,CANDIDATE_WINDOW_MISS
5,K10_M1,10,1,R2-21,Đây là câu 3 trong bài tập vận dụng. Các dữ ki...,holdout20,QA,number_or_measurement,hard,L25_V058,1,True,False,False,10,10,CANDIDATE_WINDOW_MISS
6,K10_M1,10,1,R1-24,Đoạn video về tường thuật một cuộc đua xe đạp....,holdout20,KIS,number_or_measurement,very_hard,L23_V007,2,True,False,False,10,10,CANDIDATE_WINDOW_MISS
7,K10_M3,10,3,R2-15,Một người đang chụp hình những chiếc ván trượt...,holdout20,KIS,other,medium,K01_V018,14,False,True,False,30,10,CANDIDATE_VIDEO_MISS
8,K10_M3,10,3,R2-1,Nhiều người mặt áo cờ đỏ sao vàng đứng trước b...,development20,KIS,other,easy,K03_V019,24,False,True,False,30,10,CANDIDATE_VIDEO_MISS
9,K10_M3,10,3,R2-8,Đoạn clip mở đầu với hình ảnh cây cầu có hệ th...,development20,KIS,number_or_measurement,medium,K03_V023,27,False,False,False,30,10,CANDIDATE_VIDEO_MISS


In [7]:
recommended = summary["recommended_policy_id"]
row = policies[policies["policy_id"] == recommended].iloc[0]
print("Review before Stage 7")
print(json.dumps({
    "status" : "frozen_after_review",
    "selected_video_k" : int(row["video_k"]),
    "selected_windows_per_video" : int(row["windows_per_video_m"]),
    "selected_policy_id" : recommended,
    "note" : "Freeze only after reviewing Stage 6 candidate coverage and failures."
}, indent = 2))
print("After freezing selection.stage06_decision, set the Stage 7 p90 latency ceiling before running Stage 7.")

Review before Stage 7
{
  "status": "frozen_after_review",
  "selected_video_k": 30,
  "selected_windows_per_video": 5,
  "selected_policy_id": "K30_M5",
  "note": "Freeze only after reviewing Stage 6 candidate coverage and failures."
}
After freezing selection.stage06_decision, set the Stage 7 p90 latency ceiling before running Stage 7.
